<div style="
  background: linear-gradient(145deg, #0f172a, #1e293b);
  border: 4px solid transparent;
  border-radius: 14px;
  padding: 18px 22px;
  margin: 12px 0;
  font-size: 26px;
  font-weight: 600;
  color: #f8fafc;
  box-shadow: 0 6px 14px rgba(0,0,0,0.25);
  background-clip: padding-box;
  position: relative;
">
  <div style="
    position: absolute;
    inset: 0;
    padding: 4px;
    border-radius: 14px;
    background: linear-gradient(90deg, #06b6d4, #3b82f6, #8b5cf6);
    -webkit-mask: 
      linear-gradient(#fff 0 0) content-box, 
      linear-gradient(#fff 0 0);
    -webkit-mask-composite: xor;
    mask-composite: exclude;
    pointer-events: none;
  "></div>
  <b>04. 🎨 Image Filters NumPy Vectorization</b>
</div>


### 📌 Project Overview
In this project, we develop an image filtering engine using `numpy` and `pillow` entirely from scratch.
Rather than utilizing high-level CV libraries, we implement 2D image matrix convolutions manually. We will write customized kernels (Box Blur, Sharpening, and Sobel edge detection) and leverage NumPy's vectorized slicing patterns to execute operations at native speed without slow Python loop nesting.

#### 🔑 Key Concepts Covered:
- Converting PIL images to grayscale NumPy intensity matrices
- Padded array boundaries (`np.pad`) to preserve outer dimensions during convolutions
- Matrix kernel arithmetic using parallel vectorized 2D array shifts
- Gradient magnitude estimation combining X and Y edge channels
- Comparative analysis plots in `matplotlib`


In [ ]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import urllib.request
import io

def load_demo_image():
    """Downloads a sample grayscale image, or falls back to synthetic checkerboard."""
    url = 'https://picsum.photos/id/10/400/400'
    print('Downloading sample image from Picsum...')
    try:
        headers = {'User-Agent': 'Mozilla/5.0'}
        req = urllib.request.Request(url, headers=headers)
        with urllib.request.urlopen(req, timeout=10) as response:
            img_data = response.read()
        img = Image.open(io.BytesIO(img_data)).convert('L')
        return np.array(img)
    except Exception as e:
        print(f'⚠️ Download failed ({e}). Generating synthetic checkerboard pattern.')
        canvas = np.zeros((400, 400), dtype=np.uint8)
        canvas[::40, ::40] = 255
        canvas[20::40, 20::40] = 255
        return canvas

def apply_filter_2d(image, kernel):
    """Convolves a grayscale image array with a given 2D filter kernel.
       Applies vectorized shifts in parallel to bypass Python loop overhead.
    """
    k_h, k_w = kernel.shape
    img_h, img_w = image.shape
    
    # Compute border padding sizes
    pad_h = k_h // 2
    pad_w = k_w // 2
    
    # Apply edge duplication padding
    padded = np.pad(image, ((pad_h, pad_h), (pad_w, pad_w)), mode='edge')
    
    # Accumulate element-wise shifts in floating precision
    accumulated = np.zeros_like(image, dtype=np.float32)
    for r in range(k_h):
        for c in range(k_w):
            weight = kernel[r, c]
            # Slice shifted subarray of original shape matching kernel cell position
            shifted_view = padded[r : r + img_h, c : c + img_w]
            accumulated += shifted_view * weight
            
    # Clip to valid 8-bit color space
    return np.clip(accumulated, 0, 255).astype(np.uint8)


In [ ]:
# 1. Download image context
gray_img = load_demo_image()

# 2. Define analytical matrix convolution filters
box_blur_5x5 = np.ones((5, 5), dtype=np.float32) / 25.0

sharpen_3x3 = np.array([
    [ 0, -1,  0],
    [-1,  5, -1],
    [ 0, -1,  0]
], dtype=np.float32)

sobel_horizontal = np.array([
    [-1, 0, 1],
    [-2, 0, 2],
    [-1, 0, 1]
], dtype=np.float32)

sobel_vertical = np.array([
    [-1, -2, -1],
    [ 0,  0,  0],
    [ 1,  2,  1]
], dtype=np.float32)

# 3. Execute filter conversions
print('Applying Box Blur (5x5)...')
blurred_res = apply_filter_2d(gray_img, box_blur_5x5)

print('Applying High-Pass Sharpening (3x3)...')
sharpen_res = apply_filter_2d(gray_img, sharpen_3x3)

print('Applying Sobel Edge Magnitude approximation...')
gx = apply_filter_2d(gray_img, sobel_horizontal).astype(float)
gy = apply_filter_2d(gray_img, sobel_vertical).astype(float)
edge_magnitude = np.clip(np.sqrt(gx**2 + gy**2), 0, 255).astype(np.uint8)

# 4. Render comparative plot views
plt.figure(figsize=(14, 10))

plt.subplot(2, 2, 1)
plt.imshow(gray_img, cmap='gray')
plt.title('1. Grayscale Input')
plt.axis('off')

plt.subplot(2, 2, 2)
plt.imshow(blurred_res, cmap='gray')
plt.title('2. Box Blur Filter (Smoothening)')
plt.axis('off')

plt.subplot(2, 2, 3)
plt.imshow(sharpen_res, cmap='gray')
plt.title('3. High-Pass Sharpen Filter')
plt.axis('off')

plt.subplot(2, 2, 4)
plt.imshow(edge_magnitude, cmap='gray')
plt.title('4. Sobel Edge Gradient Magnitude')
plt.axis('off')

plt.tight_layout()
plt.show()
